In [ ]:
import os
import sys

repo_path =  ".."
os.chdir(repo_path)                 # Move into the repo
sys.path.insert(0, os.getcwd())     # Ensure the repo root is on sys.path

import torch
import json
import math
from transformers import AutoTokenizer
from transformers.configuration_utils import PretrainedConfig
from transformers.utils import logging
from transformers import AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset

# Baseline DeepSeekMoE
from deepseekmoe_dynamic_routing_algorithms.source.deepseek_baseline.config import DeepseekConfig as BaselineConfig
from deepseekmoe_dynamic_routing_algorithms.source.deepseek_baseline.model import DeepseekForCausalLM as BaselineModel

# DYNMoE baseline (pure DYNMoE architecture)
from deepseekmoe_dynamic_routing_algorithms.source.DYNMoE_baseline.config import DynMoEConfig as DYNMoEBaseConfig
from deepseekmoe_dynamic_routing_algorithms.source.DYNMoE_baseline.model import DynMoEForCausalLM as DYNMoEBaseModel
from deepseekmoe_dynamic_routing_algorithms.source.DYNMoE_baseline.adaptive_tuning import AdaptiveExpertTuningCallback

# Prototype: DeepSeekMoE with DYNMoE routing
from deepseekmoe_dynamic_routing_algorithms.source.deepseek_dynamics_routing.config import DeepseekConfig as DynmoeConfig
from deepseekmoe_dynamic_routing_algorithms.source.deepseek_dynamics_routing.model import DeepseekForCausalLM as DynmoeModel
from deepseekmoe_dynamic_routing_algorithms.source.deepseek_dynamics_routing.adaptive_tuning import AdaptiveExpertTuningCallback

# Utilities
from deepseekmoe_dynamic_routing_algorithms.source.training_utils.monitoring import ResourceMonitorCallback, MoEMetricsCallback
from deepseekmoe_dynamic_routing_algorithms.source.training_utils.save_model import save_model_and_tokenizer
from deepseekmoe_dynamic_routing_algorithms.source.training_utils.summarization import print_training_summary
from deepseekmoe_dynamic_routing_algorithms.source.data_preprocessing import load_and_preprocess_multiwoz

from deepseekmoe_dynamic_routing_algorithms.source.training_utils.config import (
    MAX_SEQ_LEN,
    PER_DEVICE_BATCH,
    GRAD_ACCUM,
    LEARNING_RATE,
    NUM_EPOCHS,
    WARMUP_STEPS,
    WEIGHT_DECAY,
    EARLY_STOPPING_PATIENCE,
    EARLY_STOPPING_THRESHOLD,
    world_size
)

# Model Training Configurations
import warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

torch.cuda.empty_cache()
#world_size = torch.cuda.device_count()
print(f"Number of GPUs: {world_size}")

In [ ]:
# Shared training hyperparameters
OUTPUT_DIR_BASELINE = "/kaggle/working/deepseekmoe_dynamic_routing_algorithms/checkpoints/baseline"
OUTPUT_DIR_DYNMOE_BASE = "/kaggle/working/deepseekmoe_dynamic_routing_algorithms/checkpoints/dynmoe_baseline"
OUTPUT_DIR_DYNMOE_ROUTING = "/kaggle/working/deepseekmoe_dynamic_routing_algorithms/checkpoints/dynmoe_routing"

In [ ]:
# Load data
train_sequences, val_sequences, test_sequences = load_and_preprocess_multiwoz(
    zip_path="/kaggle/working/deepseekmoe_dynamic_routing_algorithms/dataset/MultiWOZ-coref/MultiWOZ2_3.zip",
    sample_size=300,
    random_seed=42
)

print(f"Train sequences: {len(train_sequences)}")
print(f"Validation sequences: {len(val_sequences)}")
print(f"Test sequences: {len(test_sequences)}")

In [ ]:
MAX_SEQ_LEN = 256
PER_DEVICE_BATCH = 8
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS = 1
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# DeepSpeed config
ds_config = {
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "fp16": {"enabled": True, "loss_scale": 0, "initial_scale_power": 16, "hysteresis": 2, "min_loss_scale": 1},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param": {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": 5e8,
        "stage3_prefetch_bucket_size": 5e8,
        "stage3_param_persistence_threshold": 1e6,
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9,
        "stage3_gather_16bit_weights_on_model_save": True
    },
    "gradient_clipping": 1.0,
    "steps_per_print": 10,
    "wall_clock_breakdown": False
}


with open("ds_config.json", "w") as f:
    json.dump(ds_config, f, indent=2)

In [ ]:
def train_model(ModelClass, ConfigClass, output_dir, is_dynmoe=False):
    tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-moe-16b-base", use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    config = ConfigClass()
    model = ModelClass(config)
    model.resize_token_embeddings(len(tokenizer))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    #model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    #model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model.train()
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("\nModel Summary:")
    print(f" Total parameters: {total_params:,}")
    print(f" Trainable parameters: {trainable_params:,}")
    print(f" Model architecture:\n{model}\n")
    
    dataset = load_dataset("text", data_files={"train": "train_sequences.txt", "validation": "val_sequences.txt"})
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LEN, padding=False, return_attention_mask=True)
    
    tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"], num_proc=2)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)

    train_dataset = tokenized_datasets["train"]
    eval_dataset = tokenized_datasets["validation"]
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        load_best_model_at_end=False,           # Changed to False (prototype style)
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        deepspeed="ds_config.json",   # <-- path to JSON
        ddp_find_unused_parameters=False if world_size > 1 else None,
        gradient_checkpointing=False,           # Often more stable with DeepSpeed ZeRO-3
        dataloader_num_workers=2,
        remove_unused_columns=True,
        optim="adamw_torch",
        logging_dir=f"{output_dir}/logs",
        seed=42,
        save_only_model=True                    # Important
    )
    
    resource_monitor = ResourceMonitorCallback()
    moemetrics = MoEMetricsCallback(
        eval_dataset, tokenizer, data_collator
    )
    
    callbacks = [resource_monitor, moemetrics]
    if is_dynmoe:
        adaptive_callback = AdaptiveExpertTuningCallback(audit_steps=10)
        callbacks.append(adaptive_callback)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
        callbacks=callbacks
    )
    
    print("Starting training...")
    train_result = trainer.train()
    print("Training finished.")
    
    eval_results = trainer.evaluate()
    final_loss = eval_results.get("eval_loss", float('inf'))
    perplexity = math.exp(final_loss) if 0 < final_loss < 30 else float('inf')
    print(f"Final validation loss: {final_loss:.4f}")
    print(f"Validation Perplexity: {perplexity:.2f}")
    
    save_model_and_tokenizer(trainer, output_dir)
    print_training_summary(resource_monitor, moemetrics, train_result, eval_results, perplexity)
    
    return trainer

print("Training function defined")

In [ ]:
print("Training function defined")
# Run Baseline (DeepSeekMoE)
print("=" * 60)
print("TRAINING DeepSeekMoE (BASELINE)")
print("=" * 60)
baseline_trainer = train_model(BaselineModel, BaselineConfig, OUTPUT_DIR_BASELINE, is_dynmoe=False)

In [ ]:
# Run DYNMoE Baseline (pure DYNMoE)
print("=" * 60)
print("TRAINING DYNMoE (BASELINE)")
print("=" * 60)
dynmoe_base_trainer = train_model(DYNMoEBaseModel, DYNMoEBaseConfig, OUTPUT_DIR_DYNMOE_BASE, is_dynmoe=True)

In [ ]:
# Run Prototype: DeepSeekMoE with DYNMoE routing
print("=" * 60)
print("TRAINING DeepSeekMoE DYNMoE Routing (PROTOTYPE)")
print("=" * 60)
dynmoe_routing_trainer = train_model(DynmoeModel, DynmoeConfig, OUTPUT_DIR_DYNMOE_ROUTING, is_dynmoe=True)